## Import

In [ ]:
import torch
from torch import nn
from datasets import load_dataset
import os
from math import log10, sqrt

import lpips
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as F
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset

## Set hyperparameters and device

In [ ]:
scale_factor = 2
learning_rate = 1e-3
batch_size = 64
epochs = 30

# images are split into random smaller "patches" for training. Patch size determines the size of each patch.
patch_size = 64

device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)

loss_fn_vgg = lpips.LPIPS(net="vgg").to(device)
loss_fn_pixel = nn.L1Loss().to(device)

## Load data
Load DIV2K dataset from `datasets/` following the DIV2K file structure.

In [ ]:
class Div2kDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, patch_size=None, train=True):
        """
        Args:
            lr_dir (str): Path to Low-Resolution folder
            hr_dir (str): Path to High-Resolution folder
            patch_size (int): Size of the LR patch (if cropping during training)
            train (bool): Toggle for training vs validation behaviors
        """
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.patch_size = patch_size
        self.is_train = train

        self.lr_filenames = sorted(os.listdir(lr_dir))
        self.hr_filenames = sorted(os.listdir(hr_dir))
        if not train:
            self.lr_filenames = self.lr_filenames[:30]
            self.hr_filenames = self.hr_filenames[:30]

        assert len(self.lr_filenames) == len(self.hr_filenames), (
            f"Mismatch: found {len(self.lr_filenames)} LR images but {len(self.hr_filenames)} HR images."
        )

    def __len__(self):
        return len(self.lr_filenames)

    def __getitem__(self, idx):
        lr_path = os.path.join(self.lr_dir, self.lr_filenames[idx])
        hr_path = os.path.join(self.hr_dir, self.hr_filenames[idx])
        lr_image = Image.open(lr_path).convert("RGB")
        hr_image = Image.open(hr_path).convert("RGB")

        if self.is_train and self.patch_size is not None:
            # crop images to patch size
            # calculate upscale factor
            scale = hr_image.size[0] // lr_image.size[0]

            lr_width, lr_height = lr_image.size
            # select a top-left starting coordinate on the LR image
            x = torch.randint(0, lr_width - self.patch_size + 1, (1,)).item()
            y = torch.randint(0, lr_height - self.patch_size + 1, (1,)).item()

            # crop LR image
            lr_image = F.crop(lr_image, y, x, self.patch_size, self.patch_size)
            # crop HR image to match the cropped LR image
            hr_image = F.crop(
                hr_image,
                y * scale,
                x * scale,
                self.patch_size * scale,
                self.patch_size * scale,
            )

        # transform to Pytorch Tensors
        lr_tensor = F.to_tensor(lr_image)
        hr_tensor = F.to_tensor(hr_image)

        return lr_tensor, hr_tensor


def get_datasets(patch_size, batch_size, scale_factor=2):
    training_data = Div2kDataset(
        f"datasets/DIV2K_train_LR_bicubic/X{scale_factor}",
        "datasets/DIV2K_train_HR",
        patch_size,
        train=True,
    )
    testing_data = Div2kDataset(
        f"datasets/DIV2K_valid_LR_bicubic/X{scale_factor}",
        "datasets/DIV2K_valid_HR",
        train=False,
    )

    train_dataloader = DataLoader(
        dataset=training_data,
        batch_size=batch_size,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
    )
    test_dataloader = DataLoader(
        dataset=testing_data,
        batch_size=1,
        shuffle=False,
        num_workers=2,
        pin_memory=True,
    )
    return train_dataloader, test_dataloader

train_dataloader, test_dataloader = get_datasets(
    patch_size=patch_size, batch_size=batch_size, scale_factor=scale_factor
)

## Prepare for training
- Define network layers
- Set optimizer
- Define train/test loops

In [ ]:
class UpscaleModel(nn.Module):
    """
    FSRCNN that takes in all 3 channels with a pixel shuffle layer at the end to upscale.
    """

    def __init__(self, num_channels=3, d=56, s=16, m=4, scale_factor=2):
        super().__init__()

        # feature extraction
        self.feature_extraction = nn.Sequential(
            nn.Conv2d(
                in_channels=num_channels, out_channels=d, kernel_size=5, padding=5 // 2
            ),
            nn.PReLU(d),
        )

        # shrinking
        self.shrinking = nn.Sequential(
            nn.Conv2d(d, s, kernel_size=1),
            nn.PReLU(s),
        )

        # mapping
        mapping_layers = []
        for _ in range(m):
            mapping_layers.append(nn.Conv2d(s, s, kernel_size=3, padding=3 // 2))
            mapping_layers.append(nn.PReLU(s))
        self.mapping = nn.Sequential(*mapping_layers)

        # expanding
        self.expanding = nn.Sequential(nn.Conv2d(s, d, kernel_size=1), nn.PReLU(d))

        # upscale with deconvolution + pixel shuffle
        out_channels = num_channels * scale_factor**2
        self.upscale = nn.Sequential(
            nn.Conv2d(d, out_channels, kernel_size=1),
            nn.PixelShuffle(scale_factor),
        )
        self._initialize_weights()

    def forward(self, x):
        extracted = self.feature_extraction(x)
        shrunk = self.shrinking(extracted)
        mapped = self.mapping(shrunk)
        expanded = self.expanding(mapped)
        upscaled = self.upscale(expanded)
        return upscaled

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                # Kaiming Normal balances variance across un-normalized deep structures
                nn.init.kaiming_normal_(
                    m.weight, nonlinearity="leaky_relu"
                )
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

model = UpscaleModel(scale_factor=scale_factor).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

rgb_to_grayscale_weights = torch.tensor([0.299, 0.587, 0.114]).view(1, 3, 1, 1)

def rgb_to_grayscale(rgb_image: torch.Tensor) -> torch.Tensor:
    """
    Converts RGB prediction and label tensors of shape [B, 3, X, Y] to grayscale using Rec. 601 luma weights.

    Parameters:
        rgb_image (torch.Tensor): RGB tensor of shape [B, 3, X, Y]

    Returns:
        grayscale_image (torch.Tensor): Grayscale tensor of shape [B, 1, X, Y]
    """
    # Enforce shape requirements
    if rgb_image.shape[1] != 3:
        raise ValueError("Inputs must have exactly 3 channels at dimension 1 (RGB).")
    global rgb_to_grayscale_weights
    rgb_to_grayscale_weights = rgb_to_grayscale_weights.to(
        rgb_image.device, dtype=rgb_image.dtype
    )

    grayscale_image = (rgb_image * rgb_to_grayscale_weights).sum(
        dim=1, keepdim=True
    )

    return grayscale_image

def evaluate_psnr(
    clean: torch.Tensor, degraded: torch.Tensor, max_pixel: float = 1
):
    mse = torch.mean((clean - degraded) ** 2)
    if mse == 0:
        return 100
    psnr = 20 * log10(max_pixel / sqrt(mse))
    return psnr

# train loop
def train_loop(
    dataloader: DataLoader, model, loss_fn_lpips: lpips.LPIPS, loss_fn_pixel, optimizer
):
    size = len(dataloader.dataset)
    batch_size = dataloader.batch_size
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        # compute prediction and loss
        X = X.to(device)
        y = y.to(device)
        pred = model(X)

        loss_lpips = loss_fn_lpips.forward(pred * 2 - 1, y * 2 - 1).mean()
        loss_pixel = loss_fn_pixel(pred, y)

        loss_combined = loss_lpips + loss_pixel
        # backpropagation
        loss_combined.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 10 == 0:
            print(
                f"pixel loss: {loss_pixel.item():>7f}  [{batch * batch_size + len(X):>5d}/{size:>5d}]"
            )
            print(
                f"LPIPS distance: {loss_lpips.item():>7f}  [{batch * batch_size + len(X):>5d}/{size:>5d}]"
            )
            print(
                f"loss: {loss_combined.item():>7f}  [{batch * batch_size + len(X):>5d}/{size:>5d}]"
            )


# test loop
def test_loop(dataloader, model, loss_fn_lpips: lpips.LPIPS):
    model.eval()
    num_batches = len(dataloader)
    lpips_loss = 0
    psnr = 0

    images = []
    with torch.no_grad():
        for X, y in dataloader:
            X = X.to(device)
            y = y.to(device)
            pred = model(X)
            pred = torch.clamp(pred, min=0.0, max=1.0)
            lpips_loss += loss_fn_lpips.forward(pred * 2 - 1, y * 2 - 1).mean().item()

            pred_gray = rgb_to_grayscale(pred)
            y_gray = rgb_to_grayscale(y)
            # evaluate psnr
            psnr += evaluate_psnr(pred_gray, y_gray)

            images.append(pred)
    psnr /= num_batches
    lpips_loss /= num_batches
    print(f"average LPIPS distance: {lpips_loss}")
    print(f"average PSNR: {psnr}")

    return images

## Training
Train network and save as `model.pth`

In [ ]:
def display_images(images: list[torch.Tensor]):
    for image in images:
        print(f"shape of image: {image.shape}")
        plt.imshow(image.permute(2, 3, 1, 0).cpu().numpy().squeeze())
        plt.show()

all_images = []
for t in range(epochs):
    print(f"Epoch {t + 1}\n-------------------------------")
    print("TRAINING")
    train_loop(train_dataloader, model, loss_fn_vgg, loss_fn_pixel, optimizer)
    if (t + 1) % 10 == 0 or t == epochs - 1:
        print("\nTESTING")
        all_images.append(test_loop(test_dataloader, model, loss_fn_vgg))

print("Done!")
model_file_name = "model.pth"
curr_dir = os.getcwd()
model_file_path = os.path.join(curr_dir, model_file_name)
torch.save(model.state_dict(), model_file_path)
display_images(all_images[-1])